# AI Assignment: A Classical Reasoning Agent

**Pipeline:** natural-language knowledge base → parser → facts + rules → classical reasoning engine → `True` / `False`

No generative AI model is used for predictions. The system converts natural language into first-order logical representations (ground facts and Horn-style rules) and causal structural models, and applies classical AI reasoning techniques (Forward Chaining with Unification, Stratified Negation-as-Failure, Active Domain Grounding, and Structural Causal Inference) to decide entailment.

Running this notebook top to bottom will:
1. Build and verify the classical reasoning engine on formal logic and causal structures,
2. Evaluate performance on the public benchmarks (RuleTaker, CounterBench, FOLIO) and print accuracy and inference time,
3. Generate `submission/submission.csv`, `submission/report.pdf`, and `submission/README.txt`.


In [80]:
import re, time, csv, os, json, itertools
from collections import defaultdict

# ---- Configuration ----
TEST_PATH = "private_test.csv"          # private test file (columns: id, context, question)
OUT_DIR   = "submission"        # output directory for submission files
MAX_PER_SOURCE = 1000           # maximum samples evaluated per public dataset

# Public datasets on Hugging Face (primary + robust fallbacks)
PUBLIC_SOURCES = [
    ("tasksource/ruletaker", {"split": "train"}),
    ("CounterBench/CounterBench", {"data_files": "data_balanced_alpha_V1.json", "split": "train"}),
    ("tasksource/folio", {"split": "validation"}),
]

os.makedirs(OUT_DIR, exist_ok=True)
print("Configuration loaded. Output directory:", OUT_DIR)


Configuration loaded. Output directory: submission


In [81]:
%pip install reportlab

Note: you may need to restart the kernel to use updated packages.


## 1. Representation & Parsing

* **Literals:** Represented as `(negated, predicate, args)`. E.g., `(False, "young", ("bob",))` or `(True, "smart", ("bob",))`.
* **Facts:** Ground literals containing concrete entities (constants) without variables.
* **Rules:** `body → head`, where body is a list of literals and head is a literal. Variables begin with `?` (e.g., `?X`).
* **Active Domain:** Collected constants from all entities present in the knowledge base and query.
* **Structural Causal Models:** Variables governed by causal parent equations and interventions ($do(X=\text{False})$).


In [82]:
VAR_SUBJ = {"someone", "something", "they", "it", "he", "she", "people", "things", "anyone",
            "anything", "everyone", "everything", "person", "thing"}
VAR_OBJ  = {"someone", "something", "anyone", "anything"}
PRONOUN  = {"it", "they", "he", "she"}
GENERIC  = r"(?:things|people|persons|thing|person|animals|animal|ones)"

SEED_VERBS = set("""chase eat like need see visit love hate know want fear hug kiss help follow watch
call give take push pull hit kick ride play meet admire respect trust cause lead""".split())

def stem(w):
    return w[:-1] if w.endswith("s") and not w.endswith("ss") else w

def norm(s):
    s = s.strip().lower()
    s = re.sub(r"[.!?]+$", "", s).strip()
    for a, b in (("isn't", "is not"), ("aren't", "are not"), ("doesn't", "does not"), ("don't", "do not"), ("cannot", "can not")):
        s = s.replace(a, b)
    s = re.sub(r"\b(the|a|an)\b\s*", "", s)
    return re.sub(r"\s+", " ", s).strip()

def split_and(txt):
    return [p.strip() for p in re.split(r"\s*,?\s*\band\b\s*", txt) if p.strip()]

def subj_term(s, pron):
    if s in PRONOUN:
        return pron
    if s in VAR_SUBJ:
        return "?X"
    return s.replace(" ", "_")

def obj_term(s, subj, pron):
    if s in PRONOUN:
        return pron
    if s in VAR_OBJ:
        return "?Y" if subj.startswith("?") else "?X"
    return s.replace(" ", "_")

def parse_literal(text, pron="?X"):
    t = norm(text)
    m = re.match(r"^(.+?)\s+(?:is|are|was|were|will be)\s+(not\s+)?(.+)$", t)
    if m:
        return (bool(m.group(2)), m.group(3).replace(" ", "_"), (subj_term(m.group(1), pron),))
    neg = False
    m = re.match(r"^(.+?)\s+(?:does not|do not|did not|will not)\s+(\w+)\s+(.+)$", t)
    if m:
        neg = True
        subj, verb, obj = m.group(1), stem(m.group(2)), m.group(3)
    else:
        toks = t.split()
        vi = next((i for i, w in enumerate(toks) if i > 0 and stem(w) in SEED_VERBS), None)
        if vi is None:
            vi = next((i for i, w in enumerate(toks) if i > 0 and w.endswith("s") and not w.endswith("ss")), None)
        if vi is None or vi == len(toks) - 1:
            return None
        subj, verb, obj = " ".join(toks[:vi]), stem(toks[vi]), " ".join(toks[vi + 1:])
    s = subj_term(subj, pron)
    return (neg, verb, (s, obj_term(obj, s, pron)))

def is_ground(lit):
    return not any(a.startswith("?") for a in lit[2])

def parse_clauses(txt, pron):
    out = []
    for piece in split_and(txt):
        lit = parse_literal(piece, pron)
        if lit is None and out and len(out[-1][2]) == 1:
            m = re.match(r"^(not\s+)?(.+)$", norm(piece))
            lit = (bool(m.group(1)), m.group(2).replace(" ", "_"), out[-1][2])
        out.append(lit)
        if lit is None:
            break
    return out

def parse_sentence(sent):
    t = norm(sent)
    if not t:
        return None
    # 1. Implication: If ... then ...
    m = re.match(r"^if\s+(.+?),?\s+then\s+(.+)$", t)
    if m:
        body_txt, head_txt = m.groups()
        has_var = any(re.search(r"\b%s\b" % w, body_txt + " " + head_txt) for w in VAR_OBJ | {"people", "things"})
        first = split_and(body_txt)[0]
        pron = "?X"
        if not has_var:
            lit0 = parse_literal(first)
            pron = lit0[2][0] if lit0 else "?X"
        body = parse_clauses(body_txt, pron)
        heads = parse_clauses(head_txt, pron)
        if None in body or None in heads:
            return ("fail", sent)
        return ("rule", body, heads, sent.strip())
    
    # 2. Generic quantified rules: "All/Every/Big, red things are (not) nice"
    m = re.match(r"^(?:all\s+)?(.+?)\s+%s\s+(?:are|is)\s+(not\s+)?(.+)$" % GENERIC, t)
    if m:
        conds = [c.strip() for c in re.split(r"\s*,\s*(?:and\s+)?|\s+and\s+", m.group(1)) if c.strip()]
        body = [(False, c.replace(" ", "_"), ("?X",)) for c in conds]
        head = (bool(m.group(2)), m.group(3).replace(" ", "_"), ("?X",))
        return ("rule", body, [head], sent.strip())

    # 3. Universal quantification: "All/Every X are (not) Y"
    m = re.match(r"^(?:all|every|each)\s+(.+?)\s+(?:are|is)\s+(not\s+)?(.+)$", t)
    if m:
        neg = bool(m.group(2))
        subj = m.group(1).rstrip("s").replace(" ", "_")
        pred = m.group(3).rstrip("s").replace(" ", "_")
        return ("rule", [(False, subj, ("?X",))], [(neg, pred, ("?X",))], sent.strip())

    # 4. Universal negative: "No X is/are Y"
    m = re.match(r"^no\s+(.+?)\s+(?:are|is)\s+(.+)$", t)
    if m:
        subj = m.group(1).rstrip("s").replace(" ", "_")
        pred = m.group(2).rstrip("s").replace(" ", "_")
        return ("rule", [(False, subj, ("?X",))], [(True, pred, ("?X",))], sent.strip())

    # 5. Simple Fact
    lit = parse_literal(t)
    if lit is None or not is_ground(lit):
        return ("fail", sent)
    return ("fact", lit)

def split_sentences(context):
    parts = []
    for line in re.split(r"\n+", context.strip()):
        parts += re.split(r"(?<=[.!?])\s+", line.strip())
    return [p for p in parts if p.strip()]

class Rule:
    def __init__(self, body, head, text):
        self.body = sorted(body, key=lambda l: l[0])
        self.head = head
        self.text = text

def build_kb(context):
    facts, rules, failures = [], [], []
    for s in split_sentences(context):
        res = parse_sentence(s)
        if res is None:
            continue
        kind = res[0]
        if kind == "fact":
            facts.append(res[1])
        elif kind == "rule":
            for h in res[2]:
                rules.append(Rule(res[1], h, res[3]))
        else:
            failures.append(s)
    return facts, rules, failures


## 2. Classical Reasoning Engine

The reasoning engine implements:
1. **Forward Chaining with Unification:** Matches rule antecedents against the fact database, binding variables via syntactic unification.
2. **Stratification & Negation-As-Failure (NAF):** Computes dependency strata so that negative premises are only checked once their positive base is completely derived.
3. **Active Domain Grounding:** In classical logic, rules where variables appear in negative literals (e.g. `If someone is not kind then they are quiet`) must range over the active domain of entities in the knowledge base. The engine grounds unbound variables across all known domain constants.
4. **Backward Relevance Filtering:** Prunes rules whose conclusions cannot reach the query predicate.
5. **Structural Causal Evaluation:** Solves causal counterfactual queries ($do(X=\text{False})$) through structural equation graph evaluation.


In [83]:
def unify(pat, vals, theta):
    if len(pat) != len(vals):
        return None
    t, copied = theta, False
    for p, v in zip(pat, vals):
        if p[0] == "?":
            if p in t:
                if t[p] != v:
                    return None
            else:
                if not copied:
                    t, copied = dict(t), True
                t[p] = v
        elif p != v:
            return None
    return t

def subst(args, theta):
    return tuple(theta.get(a, a) if a[0] == "?" else a for a in args)

def join(body, i, theta, used, idx, cwa):
    if i == len(body):
        yield theta, used
        return
    neg, pred, args = body[i]
    if neg and cwa:
        for fargs in idx.get((False, pred), ()):
            if unify(args, fargs, theta) is not None:
                return
        yield from join(body, i + 1, theta, used + [("naf", pred, subst(args, theta))], idx, cwa)
        return
    for fargs in list(idx.get((neg, pred), ())):
        t2 = unify(args, fargs, theta)
        if t2 is not None:
            yield from join(body, i + 1, t2, used + [(neg, pred, fargs)], idx, cwa)

def relevant_rules(rules, goal_pred):
    needed, changed = {goal_pred}, True
    while changed:
        changed = False
        for r in rules:
            if r.head[1] in needed:
                for _, p, _ in r.body:
                    if p not in needed:
                        needed.add(p); changed = True
    return [r for r in rules if r.head[1] in needed]

def stratify(rules, cwa):
    st = defaultdict(int)
    limit = len(rules) + 2
    for _ in range(limit * 2):
        changed = False
        for r in rules:
            for neg, p, _ in r.body:
                need = st[p] + (1 if (neg and cwa) else 0)
                if st[r.head[1]] < need:
                    st[r.head[1]] = need; changed = True
        if not changed:
            return {id(r): st[r.head[1]] for r in rules}
    return {id(r): 0 for r in rules}

def ground_rules_over_domain(rules, constants):
    grounded = []
    for r in rules:
        pos_vars = set()
        for neg, _, args in r.body:
            if not neg:
                for a in args:
                    if a.startswith("?"):
                        pos_vars.add(a)
        all_v = set()
        for _, _, args in r.body:
            for a in args:
                if a.startswith("?"):
                    all_v.add(a)
        for a in r.head[2]:
            if a.startswith("?"):
                all_v.add(a)

        unbound_vars = sorted(all_v - pos_vars)
        if not unbound_vars or not constants:
            grounded.append(r)
        else:
            for combo in itertools.product(constants, repeat=len(unbound_vars)):
                mapping = dict(zip(unbound_vars, combo))
                new_body = [(neg, pred, subst(args, mapping)) for neg, pred, args in r.body]
                new_head = (r.head[0], r.head[1], subst(r.head[2], mapping))
                grounded.append(Rule(new_body, new_head, r.text))
    return grounded

def forward_chain(facts, rules, constants=None, cwa=True):
    prov, idx = {}, defaultdict(set)
    def add(f, why):
        if f in prov:
            return False
        prov[f] = why
        idx[(f[0], f[1])].add(f[2])
        return True

    for f in facts:
        add(f, ("given",))

    rules_to_run = ground_rules_over_domain(rules, constants or set())
    layer = stratify(rules_to_run, cwa)
    for level in sorted(set(layer.values())):
        todo = [r for r in rules_to_run if layer[id(r)] == level]
        changed = True
        while changed:
            changed = False
            for r in todo:
                for theta, used in list(join(r.body, 0, {}, [], idx, cwa)):
                    h = subst(r.head[2], theta)
                    if any(a[0] == "?" for a in h):
                        continue
                    if add((r.head[0], r.head[1], h), ("rule", r.text, used)):
                        changed = True
    return prov

def fmt(l):
    if l[0] == "naf":
        return "not %s(%s) [cannot be derived]" % (l[1], ",".join(l[2]))
    return "%s%s(%s)" % ("not " if l[0] else "", l[1], ",".join(l[2]))

def explain(fact, prov, depth=0, seen=None):
    seen = seen if seen is not None else set()
    pad = "  " * depth
    why = prov.get(fact)
    if why is None:
        return [pad + fmt(fact) + "   <- not derivable"]
    if why[0] == "given" or fact in seen:
        return [pad + fmt(fact) + "   [given]"]
    seen.add(fact)
    lines = [pad + fmt(fact) + "   <- " + why[1]]
    for p in why[2]:
        lines += explain(p, prov, depth + 1, seen) if p[0] != "naf" else [pad + "  " + fmt(p)]
    return lines

# Structural Causal Reasoning for Counterfactuals
def eval_counterbench(given_info, question, trace=False):
    text = re.sub(r"^We know that\s+", "", given_info.strip(), flags=re.I).rstrip(".")
    text = re.sub(r",\s*and\s+", ", ", text)
    pattern = r"\.\s*|,\s*(?=(?:(?:not\s+)?[A-Za-z0-9_]+(?:\s+(?:or|and)\s+(?:not\s+)?[A-Za-z0-9_]+)*\s+(?:together\s+)?causes?|We observed))"
    raw_stmts = re.split(pattern, text, flags=re.I)

    equations, observations = {}, {}
    for cl in raw_stmts:
        cl = cl.strip().rstrip(".")
        if not cl or "~" in cl:
            continue
        m_obs = re.match(r"We observed\s+(not\s+\w+|\w+)", cl, re.I)
        if m_obs:
            obs_val = m_obs.group(1).strip()
            observations[obs_val[4:].strip() if obs_val.lower().startswith("not ") else obs_val] = not obs_val.lower().startswith("not ")
            continue
        m = re.match(r"^(.*?)\s+(?:together\s+)?causes?\s+(.*?)$", cl, re.I)
        if not m:
            continue
        lhs, rhs = m.group(1).strip(), m.group(2).strip()
        rhs_parts = re.split(r",\s*(?:and\s+)?|\s+and\s+", rhs) if not rhs.lower().startswith("not ") else [rhs]
        for rt in rhs_parts:
            rt = rt.strip()
            if rt:
                equations[rt] = lhs

    interventions = {}
    m1 = re.match(r"Would\s+(\w+)\s+occur\s+if\s+(.*?)(?:\s+instead of\s+.*)?\?", question, re.I)
    if m1:
        query_var, cond = m1.group(1).strip(), m1.group(2).strip()
        for part in cond.split(" and "):
            part = part.strip()
            m_neg = re.match(r"not\s+(\w+)", part, re.I)
            if m_neg:
                interventions[m_neg.group(1)] = False
            else:
                interventions[part] = True
        res = eval_cb_node(query_var, equations, interventions, observations, {})
        return (res, ["Causal structural equation evaluation for %s: %s" % (query_var, res)]) if trace else res

    m3 = re.match(r"Assume\s+(not\s+\w+|\w+),\s+and based on this assumption,\s+further suppose\s+(not\s+\w+|\w+)\.\s+Would\s+(\w+)\s+occur\?", question, re.I)
    if m3:
        p1, p2, query_var = m3.group(1).strip(), m3.group(2).strip(), m3.group(3).strip()
        for p in [p1, p2]:
            m_neg = re.match(r"not\s+(\w+)", p, re.I)
            if m_neg:
                interventions[m_neg.group(1)] = False
            else:
                interventions[p] = True
        res = eval_cb_node(query_var, equations, interventions, observations, {})
        return (res, ["Causal structural equation evaluation for %s: %s" % (query_var, res)]) if trace else res

    return None

def eval_cb_node(var, equations, interventions, observations, memo):
    if var in memo:
        return memo[var]
    if var in interventions:
        memo[var] = interventions[var]
        return memo[var]
    neg_key = f"not {var}"
    if neg_key in equations:
        val = eval_cb_expr(equations[neg_key], equations, interventions, observations, memo)
        memo[var] = not val
        return not val
    if var in equations:
        res = eval_cb_expr(equations[var], equations, interventions, observations, memo)
        memo[var] = res
        return res
    if var in observations:
        memo[var] = observations[var]
        return memo[var]
    return True

def eval_cb_expr(expr, equations, interventions, observations, memo):
    expr = expr.strip()
    if " or " in expr:
        return any(eval_cb_expr(p, equations, interventions, observations, memo) for p in expr.split(" or "))
    elif " and " in expr:
        return all(eval_cb_expr(p, equations, interventions, observations, memo) for p in expr.split(" and "))
    elif expr.lower().startswith("not "):
        return not eval_cb_node(expr[4:].strip(), equations, interventions, observations, memo)
    else:
        return eval_cb_node(expr, equations, interventions, observations, memo)

def entails(context, query, cwa=True, trace=False):
    # Route CounterBench causal structural equations
    if "causes" in context or "cause" in context or ("Would" in query and "occur" in query):
        cb_res = eval_counterbench(context, query, trace=trace)
        if cb_res is not None:
            return cb_res

    facts, rules, _ = build_kb(context)
    q = parse_literal(query)
    if q is None or not is_ground(q):
        return (False, ["could not parse the query"]) if trace else False

    consts = set()
    for f in facts:
        for a in f[2]:
            if not a.startswith("?"):
                consts.add(a)
    for r in rules:
        for _, _, args in r.body:
            for a in args:
                if not a.startswith("?"):
                    consts.add(a)
        for a in r.head[2]:
            if not a.startswith("?"):
                consts.add(a)
    for a in q[2]:
        if not a.startswith("?"):
            consts.add(a)

    prov = forward_chain(facts, relevant_rules(rules, q[1]), constants=consts, cwa=cwa)
    pos = (False, q[1], q[2])
    if q in prov:
        ans, why = True, explain(q, prov)
    elif q[0] and cwa and pos not in prov:
        ans, why = True, ["not %s(%s) holds because it cannot be derived (closed-world assumption)" % (q[1], ",".join(q[2]))]
    else:
        ans, why = False, ["%s is not derivable from the knowledge base" % fmt(q)]
    return (ans, why) if trace else ans


## 3. Sanity Checks & Reasoning Traces

Verifying the reasoning system on canonical benchmarks, including the assignment problem sheet example and negation-as-failure.


In [84]:
DEMO = [
 ("Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.", "Bob is successful.", True),
 ("Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.", "Bob is not smart.", False),
 ("The cat chases the dog. The dog is big. If something chases the dog then it is angry.", "The cat is angry.", True),
 ("Alan is big. Alan is red. Big, red things are nice.", "Alan is nice.", True),
 ("Alan is big. Big, red things are nice.", "Alan is nice.", False),
 ("Fiona is young. Erin is young. Erin is kind. If someone is young and not kind then they are rude.", "Fiona is rude.", True),
 ("Fiona is young. Erin is young. Erin is kind. If someone is young and not kind then they are rude.", "Erin is rude.", False),
 ("Bob is rough. If someone is rough then they are not nice.", "Bob is not nice.", True),
 ("The bear is big. If the bear is big then the bear is red. If the bear is red then the bear needs the cat.", "The bear needs the cat.", True),
 ("The bear is cold. If the bear is cold and it is nice then it is green.", "The bear is green.", False),
]

for i, (c, q, y) in enumerate(DEMO):
    got = entails(c, q, cwa=True)
    status = "ok " if got == y else "FAIL"
    print("%s %2d %-40s -> %s" % (status, i, q, got))

ans, why = entails(DEMO[0][0], DEMO[0][1], trace=True)
print("\nReasoning trace for Problem Sheet Example (answer = %s):" % ans)
print("\n".join(why))


ok   0 Bob is successful.                       -> True
ok   1 Bob is not smart.                        -> False
ok   2 The cat is angry.                        -> True
ok   3 Alan is nice.                            -> True
ok   4 Alan is nice.                            -> False
ok   5 Fiona is rude.                           -> True
ok   6 Erin is rude.                            -> False
ok   7 Bob is not nice.                         -> True
ok   8 The bear needs the cat.                  -> True
ok   9 The bear is green.                       -> False

Reasoning trace for Problem Sheet Example (answer = True):
successful(bob)   <- If someone is smart and kind, then they are successful.
  smart(bob)   <- If someone is young, then they are smart.
    young(bob)   [given]
  kind(bob)   [given]


## 4. Benchmark Dataset Loading

Loading and standardizing the three target benchmark datasets: **RuleTaker**, **CounterBench**, and **FOLIO**.


In [85]:
def pick(d, names):
    for n in names:
        if n in d and d[n] is not None:
            return d[n]
    return None

def to_bool(x):
    return str(x).strip().lower() in {"true", "entailment", "entailed", "yes", "1", "t"}

def as_text(x):
    return " ".join(x) if isinstance(x, (list, tuple)) else str(x)

def normalise_row(d, i):
    ctx = pick(d, ["context", "premises", "theory", "story", "facts", "text", "given_info"])
    qs  = pick(d, ["question", "statement", "conclusion", "query", "hypothesis"])
    lab = pick(d, ["label", "answer", "flag", "gold"])
    if ctx is None or qs is None:
        return None
    return {
        "id": pick(d, ["id", "ID", "idx", "question_id"]) if pick(d, ["id", "ID", "idx", "question_id"]) is not None else i,
        "context": as_text(ctx),
        "question": as_text(qs),
        "label": None if lab is None else to_bool(lab)
    }

def load_public():
    data = {}
    try:
        from datasets import load_dataset
    except Exception as e:
        print("`datasets` library not available (%s); using the built-in demo set." % type(e).__name__)
        return {"demo": [{"id": i, "context": c, "question": q, "label": y} for i, (c, q, y) in enumerate(DEMO)]}

    for name, kwargs in PUBLIC_SOURCES:
        try:
            ds = load_dataset(name, **kwargs)
            rows = [normalise_row(dict(r), i) for i, r in enumerate(ds)]
            rows = [r for r in rows if r and r["label"] is not None][:MAX_PER_SOURCE]
            data[name] = rows
            print("Loaded %-28s: %d examples" % (name, len(rows)))
        except Exception as e:
            print("Skipped %-27s: (%s)" % (name, str(e)[:80]))

    return data or {"demo": [{"id": i, "context": c, "question": q, "label": y} for i, (c, q, y) in enumerate(DEMO)]}

public = load_public()


Loaded tasksource/ruletaker        : 1000 examples
Loaded CounterBench/CounterBench   : 1000 examples
Loaded tasksource/folio            : 203 examples


## 5. Experiments on Public Benchmarks (Accuracy & Inference Time)


In [86]:
def evaluate(rows, cwa):
    t0 = time.perf_counter()
    hits, parse_fail = 0, 0
    for r in rows:
        try:
            pred = entails(r["context"], r["question"], cwa=cwa)
        except Exception:
            pred, parse_fail = False, parse_fail + 1
        hits += (pred == r["label"])
    dt = time.perf_counter() - t0
    return 100.0 * hits / max(1, len(rows)), dt, parse_fail

results = {}
for name, rows in public.items():
    for cwa in (True, False):
        acc, dt, pf = evaluate(rows, cwa)
        results[(name, cwa)] = (acc, dt, len(rows), pf)
        print("%-28s CWA=%-5s accuracy=%6.2f%%  time=%7.3fs  (%d queries, %d crashes)" % (name, cwa, acc, dt, len(rows), pf))

def avg(cwa):
    v = [results[(n, cwa)][0] for n in public]
    return sum(v) / len(v)

CWA = avg(True) >= avg(False)
print("\nChosen Global Setting: CWA =", CWA)


tasksource/ruletaker         CWA=True  accuracy=100.00%  time=  0.249s  (1000 queries, 0 crashes)
tasksource/ruletaker         CWA=False accuracy= 72.30%  time=  0.241s  (1000 queries, 0 crashes)
CounterBench/CounterBench    CWA=True  accuracy= 94.50%  time=  0.024s  (1000 queries, 0 crashes)
CounterBench/CounterBench    CWA=False accuracy= 94.50%  time=  0.023s  (1000 queries, 0 crashes)
tasksource/folio             CWA=True  accuracy= 60.10%  time=  0.029s  (203 queries, 0 crashes)
tasksource/folio             CWA=False accuracy= 64.53%  time=  0.029s  (203 queries, 0 crashes)

Chosen Global Setting: CWA = True


## 6. Execution on Test Set & Submission CSV Generation


In [87]:
# def read_test(path):
#     import pandas as pd
#     df = pd.read_csv(path)
#     rows = []
#     for i, r in df.iterrows():
#         n = normalise_row(dict(r), i)
#         if n:
#             rows.append(n)
#     return rows

# if os.path.exists(TEST_PATH):
#     test_rows = read_test(TEST_PATH)
#     print("Loaded %d test queries from %s" % (len(test_rows), TEST_PATH))
# else:
#     print("!! %s not found. Using public demo queries for submission format verification." % TEST_PATH)
#     test_rows = [{"id": i, "context": c, "question": q, "label": y} for i, (c, q, y) in enumerate(DEMO)]

# t0 = time.perf_counter()
# preds = []
# for r in test_rows:
#     ans = entails(r["context"], r["question"], cwa=CWA)
#     preds.append((r["id"], ans))
# infer_time = time.perf_counter() - t0

# out_csv = os.path.join(OUT_DIR, "submission.csv")
# with open(out_csv, "w", newline="") as f:
#     w = csv.writer(f)
#     w.writerow(["id", "label"])
#     for i, p in preds:
#         w.writerow([i, p])

# print("Wrote %s (%d rows)" % (out_csv, len(preds)))
# print("Total inference time: %.3f s (%.2f ms per query)" % (infer_time, 1000 * infer_time / max(1, len(preds))))

# # Headline numbers
# per_ds_acc  = [results[(n, CWA)][0] for n in public]
# per_ds_time = [results[(n, CWA)][1] / max(1, results[(n, CWA)][2]) for n in public]
# print("\nSummary across benchmark datasets (CWA=%s):" % CWA)
# for n, a, t in zip(public, per_ds_acc, per_ds_time):
#     print("  %-28s accuracy=%6.2f%%   avg time/sample=%.6f s" % (n, a, t))

# final_acc  = sum(per_ds_acc) / len(per_ds_acc)
# final_time = sum(per_ds_time) / len(per_ds_time)
# print("\nOverall Accuracy (macro-average over datasets):        %.2f%%" % final_acc)
# print("Overall Inference Time (avg sec/sample over datasets): %.6f s" % final_time)


# ============================================
# NEW CSV: OVERALL ACCURACY AND INFERENCE TIME
# ============================================

def read_test(path):
    import pandas as pd
    df = pd.read_csv(path)
    rows = []

    for i, r in df.iterrows():
        n = normalise_row(dict(r), i)
        if n:
            rows.append(n)

    return rows


# Check and load the actual CSV
if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(
        f"{TEST_PATH} not found. Check the CSV path."
    )

test_rows = read_test(TEST_PATH)

if len(test_rows) == 0:
    raise ValueError("No valid test queries found.")

print("Loaded %d queries from %s" % (len(test_rows), TEST_PATH))


# Run reasoning and measure inference time
t0 = time.perf_counter()

preds = []

for r in test_rows:
    ans = entails(r["context"], r["question"], cwa=CWA)
    preds.append((r["id"], ans))

infer_time = time.perf_counter() - t0


# Save predictions
os.makedirs(OUT_DIR, exist_ok=True)

out_csv = os.path.join(OUT_DIR, "submission.csv")

with open(out_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "label"])

    for i, p in preds:
        w.writerow([i, p])


# Calculate accuracy using actual CSV labels
actual_labels = [r.get("label") for r in test_rows]
predicted_labels = [p for _, p in preds]

print("\n========== FINAL RESULTS ==========")
print("Dataset:", TEST_PATH)
print("Total Queries:", len(test_rows))

if all(label is not None for label in actual_labels):

    correct = sum(
        str(a).strip().lower() == str(p).strip().lower()
        for a, p in zip(actual_labels, predicted_labels)
    )

    accuracy = (correct / len(test_rows)) * 100

    print("Correct Predictions:", correct)
    print("Overall Accuracy: %.2f%%" % accuracy)

else:
    print("Overall Accuracy: N/A (actual labels missing)")

print("Total Inference Time: %.6f seconds" % infer_time)

print("Average Inference Time: %.6f seconds/query" %
      (infer_time / len(test_rows)))

print("Prediction file:", out_csv)
print("===================================")



Loaded 100 queries from private_test.csv

========== FINAL RESULTS ==========
Dataset: private_test.csv
Total Queries: 100
Correct Predictions: 61
Overall Accuracy: 61.00%
Total Inference Time: 0.012929 seconds
Average Inference Time: 0.000129 seconds/query
Prediction file: submission/submission.csv


## 7. Report PDF & README Generation


In [88]:
def trace_text(i):
    return "\n".join(entails(DEMO[i][0], DEMO[i][1], cwa=True, trace=True)[1])

exp_lines = []
for (name, cwa), (acc, dt, n, pf) in results.items():
    exp_lines.append("%s | CWA=%s | %d queries | accuracy %.2f%% | %.3f s" % (name, cwa, n, acc, dt))

REPORT = [
 ("1. Representation",
  "Every problem is mapped to formal logical representations without any generative AI. "
  "Natural language assertions are converted to first-order Horn logic literals of the form (negated, predicate, arguments). "
  "Facts are ground literals with concrete entities (e.g. young(bob)). Rules are implication clauses of the form body -> head. "
  "Universal quantifiers, pronouns, and indefinite references (e.g. someone, something, all, every) map to logical variables (?X, ?Y). "
  "The active domain is dynamically extracted from all constants occurring in the knowledge base and query. "
  "For causal counterfactuals (CounterBench), assertions are represented as structural causal models with parent equations and intervention variables."),
 ("2. Reasoning algorithms",
  "Inference is performed strictly using classical AI reasoning techniques: "
  "(1) Forward Chaining with Unification: rule bodies are matched against the fact base using syntactic unification with backtracking over variable substitutions. "
  "(2) Active Domain Grounding: rules with variables in negative literals (e.g. not kind(?X) -> quiet(?X)) are grounded over the active domain of entities to ensure range restriction and complete coverage. "
  "(3) Stratified Negation-As-Failure: predicate dependency graphs are partitioned into strata so that negative premises are evaluated only after positive strata have completely stabilized. "
  "(4) Backward Relevance Pruning: rules unconnected to the query predicate in the dependency graph are pruned prior to inference. "
  "(5) Structural Causal Model Evaluation: counterfactual interventions (do(X=False)) propagate deterministically through causal DAG equations."),
 ("3. Experiments on the public datasets",
  "\n".join(exp_lines) + "\n\n"
  "Across the three mandated Hugging Face benchmarks (RuleTaker, CounterBench, and FOLIO):\n"
  "- RuleTaker: achieved 100.00%% accuracy due to active domain grounding and stratified negation-as-failure.\n"
  "- CounterBench: achieved 94.50%% accuracy through structural causal equation graph evaluation.\n"
  "- FOLIO: achieved 64.53%% accuracy under open-world deduction.\n"
  "Overall macro-average accuracy: %.2f%% across benchmarks with average inference time of %.6f s per query." % (final_acc, final_time)),
 ("4. Example reasoning traces",
  "Example A (Problem sheet example): Bob is young, Bob is kind; young -> smart; smart and kind -> successful. Query: Bob is successful.\n"
  + trace_text(0) +
  "\n\nExample B (Stratified negation as failure): Fiona is young; Erin is young and kind; young and not kind -> rude. Query: Fiona is rude.\n"
  + trace_text(5)),
]

README = """README
======
Name: Patani Kalp Bharatkumar
Roll no: M26CSA009

Files
  reasoning_agent.ipynb : full implementation (parsers, classical reasoning engine, evaluation)
  submission.csv        : predictions (id,label)
  report.pdf            : short report (representation, algorithms, experiments, traces)

How to run
  1. pip install datasets pandas reportlab
  2. Put the test file next to the notebook as test.csv (columns: id, context, question)
     or change TEST_PATH in the first cell.
  3. Run all cells. The last cells print accuracy and inference time and write the three
     files above into the submission/ folder.

Method in one line
  Formal Logic Parser + SCM -> Forward Chaining with Unification, Stratified NAF,
  Active Domain Grounding, and Structural Causal Inference -> True/False.
  No generative AI is used for predictions.
"""

with open(os.path.join(OUT_DIR, "README.txt"), "w") as f:
    f.write(README)

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted
    ss = getSampleStyleSheet()
    mono = ParagraphStyle("mono", parent=ss["Code"], fontSize=7.5, leading=9)
    story = [
        Paragraph("AI Assignment: Classical Reasoning Agent", ss["Title"]),
        Paragraph("Patani Kalp Bharatkumar | M26CSA009", ss["Normal"]),
        Spacer(1, 8)
    ]
    def esc(s): return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    for head, text in REPORT:
        story.append(Paragraph(head, ss["Heading2"]))
        parts = text.split("\n")
        for ln in parts:
            if ln.startswith(" ") or ("(" in ln and "<-" in ln):
                story.append(Preformatted(ln, mono))
            elif ln.strip():
                story.append(Paragraph(esc(ln), ss["BodyText"]))
        story.append(Spacer(1, 6))
    SimpleDocTemplate(os.path.join(OUT_DIR, "report.pdf"), pagesize=A4).build(story)
    print("Wrote report.pdf and README.txt in", OUT_DIR)
except Exception as e:
    with open(os.path.join(OUT_DIR, "report.txt"), "w") as f:
        f.write("\n\n".join(h + "\n" + t for h, t in REPORT))
    print("Wrote report.txt (reportlab error: %s)" % e)


Wrote report.pdf and README.txt in submission
